# Broadcast → tactical: автоматическое сопоставление игровых планов
Исходные геометрические данные и пары совпадений остаются локальными. Это анализ синхронизации, не модель D.

In [ ]:
from pathlib import Path
import sys,json,time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.neighbors import NearestNeighbors
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'games.py').exists())
sys.path.insert(0,str(ROOT))
from games import asset_paths,half_video_blocks,GAME
from starter.baseline import team_of_each_track
OUT=ROOT/'local_review/broadcast_alignment'
FPS=25;STEP=25
PITCH=np.array([105.3,68.])
arrays={}
for source in ['tactical','broadcast']:
    paths=asset_paths(source=source)
    team=team_of_each_track(paths['atd_meta'])
    h=pd.read_csv(paths['homography'])
    frames=np.arange(0,len(h),STEP)
    players=np.full((len(frames),len(team),2),np.nan,dtype=np.float32)
    ball=np.full((len(frames),2),np.nan,dtype=np.float32)
    # Sample geometry, never decode or transmit video frames.
    # Выбираем геометрию; кадры видео не декодируются и не передаются.
    with open(paths['atd_raw']) as f:
        for i,line in enumerate(tqdm(f,total=len(h),desc=source)):
            if i%STEP:continue
            fid,p,b=line.strip().split(':');assert int(fid)==i+1
            players[i//STEP]=np.fromstring(p.replace(';',','),sep=',').reshape(-1,3)[:,:2]*PITCH
            ball[i//STEP]=np.fromstring(b,sep=',')[:2]*PITCH
    valid=h.valid.eq(1).to_numpy()[frames]
    seen=np.isfinite(players).all(-1)
    arrays[source]=dict(frames=frames,players=players,ball=ball,team=team,valid=valid,
        seen=seen,homography_valid=h.valid.eq(1).to_numpy())
    np.savez(OUT/(source+'_sampled_geometry.npz'),**arrays[source])
print('Geometry sampled locally')


In [ ]:
# Candidate retrieval by ball position, followed by team-specific spatial matching.
# Кандидаты по мячу, затем проверка расстановки каждой команды без идентичности слотов.
t=arrays['tactical'];b=arrays['broadcast']
from starter.baseline import team_names
assert team_names(asset_paths(source='tactical')['atd_meta'])==team_names(asset_paths(source='broadcast')['atd_meta'])
for d in [t,b]:
    d['usable']=d['seen']&(d['players']>=-2).all(-1)&(d['players']<=PITCH+2).all(-1)
    d['eligible']=d['valid']&np.isfinite(d['ball']).all(1)&(d['ball']>=0).all(1)&(d['ball']<=PITCH).all(1)
    for team in [0,1]:d['eligible']&=(d['usable'][:,d['team']==team].sum(1)>=3)
ti=np.flatnonzero(t['eligible']);bi=np.flatnonzero(b['eligible'])
knn=NearestNeighbors(n_neighbors=min(100,len(ti))).fit(t['ball'][ti])
_,candidates=knn.kneighbors(b['ball'][bi])
records=[]
for j,ib in enumerate(tqdm(bi,desc='Match live geometry / Сопоставление геометрии')):
    it=ti[candidates[j]]
    costs=[];supports=[]
    for team in [0,1]:
        bp=b['players'][ib,b['usable'][ib]&(b['team']==team)]
        tp=t['players'][it][:,t['team']==team]
        ok=t['usable'][it][:,t['team']==team]
        distances=np.linalg.norm(tp[:,:,None,:]-bp[None,None,:,:],axis=-1)
        distances[~ok]=np.inf
        nearest=distances.min(axis=1)
        costs.append(np.median(nearest,axis=1));supports.append((nearest<=3.).mean(1))
    ball_distance=np.linalg.norm(t['ball'][it]-b['ball'][ib],axis=1)
    spatial=(costs[0]+costs[1])/2
    score=spatial+.25*ball_distance
    best=int(score.argmin());winner=int(it[best])
    far=abs(t['frames'][it]-t['frames'][winner])>10*FPS
    alternative=float(score[far].min()) if far.any() else float('inf')
    record=dict(broadcast_frame=int(b['frames'][ib]),tactical_frame=int(t['frames'][winner]),
        score=float(score[best]),spatial_m=float(spatial[best]),ball_distance_m=float(ball_distance[best]),
        support=float(min(supports[0][best],supports[1][best])),alternative_score=alternative)
    record['strong']=(record['spatial_m']<=2.5 and record['ball_distance_m']<=5 and record['support']>=.6
        and record['alternative_score']>=record['score']+1.)
    records.append(record)
matches=pd.DataFrame(records)
matches.to_csv(OUT/'coarse_content_matches.csv',index=False)
strong=matches[matches.strong].copy()
strong['offset_seconds']=(strong.tactical_frame-strong.broadcast_frame)/FPS
hist=strong.offset_seconds.round().value_counts()
summary=dict(broadcast_candidate_seconds=len(bi),tactical_candidate_seconds=len(ti),
             strong_content_matches=len(strong),top_offset_bins=[{'offset_s':float(k),'matches':int(v)} for k,v in hist.head(8).items()])
(OUT/'coarse_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))


In [ ]:
# Refine independent samples on the original 25-fps tactical geometry.
# Уточняем выбранные совпадения по исходной тактической геометрии 25 FPS.
strong=matches[matches.strong].copy()
strong['half']=0
for half,first,last in half_video_blocks(GAME):
    strong.loc[strong.tactical_frame.between(first,last),'half']=half
strong['offset_frames']=strong.tactical_frame-strong.broadcast_frame
selected=[];coarse_centers={}
for half in [1,2]:
    group=strong[strong.half==half]
    # Robust median rejects the minority of matches to old moments shown in replays.
    # Медиана отбрасывает меньшинство совпадений с прошлыми эпизодами в повторах.
    center=float(group.offset_frames.median());coarse_centers[half]=center
    group=group[abs(group.offset_frames-center)<=2*FPS].sort_values('broadcast_frame')
    previous=-100000
    for row in group.itertuples():
        if row.broadcast_frame-previous>=5*FPS:
            selected.append(row);previous=row.broadcast_frame
paths=asset_paths(source='tactical')
full_tplayers=np.full((len(t['homography_valid']),len(t['team']),2),np.nan,dtype=np.float32)
full_tball=np.full((len(full_tplayers),2),np.nan,dtype=np.float32)
with open(paths['atd_raw']) as f:
    for i,line in enumerate(tqdm(f,total=len(full_tplayers),desc='25fps reference geometry')):
        frame,p,ball=line.strip().split(':')
        full_tplayers[i]=np.fromstring(p.replace(';',','),sep=',').reshape(-1,3)[:,:2]*PITCH
        full_tball[i]=np.fromstring(ball,sep=',')[:2]*PITCH
refined=[]
for row in tqdm(selected,desc='Refine anchors / Уточнение якорей'):
    ib=row.broadcast_frame//STEP
    candidates=np.arange(max(0,row.tactical_frame-FPS),min(len(full_tplayers),row.tactical_frame+FPS+1))
    costs=[];supports=[]
    for team in [0,1]:
        bp=b['players'][ib,b['usable'][ib]&(b['team']==team)]
        tp=full_tplayers[candidates][:,t['team']==team]
        good=np.isfinite(tp).all(-1)&(tp>=-2).all(-1)&(tp<=PITCH+2).all(-1)
        dist=np.linalg.norm(tp[:,:,None]-bp[None,None],axis=-1)
        dist[~good]=np.inf
        nearest=dist.min(1)
        costs.append(np.median(nearest,axis=1));supports.append((nearest<=3.).mean(1))
    spatial=(costs[0]+costs[1])/2
    ball_dist=np.linalg.norm(full_tball[candidates]-b['ball'][ib],axis=1)
    cost=spatial+.25*ball_dist;cost[~np.isfinite(cost)]=np.inf
    win=int(cost.argmin())
    if not np.isfinite(cost[win]):continue
    refined.append(dict(broadcast_frame=int(row.broadcast_frame),tactical_frame=int(candidates[win]),half=int(row.half),
        offset_frames=int(candidates[win]-row.broadcast_frame),score=float(cost[win]),
        spatial_m=float(spatial[win]),support=float(min(supports[0][win],supports[1][win]))))
anchors=pd.DataFrame(refined).sort_values('broadcast_frame')
anchors.to_csv(OUT/'refined_anchors.csv',index=False)
models=[]
for half in [1,2]:
    group=anchors[anchors.half==half].copy()
    train=group.iloc[::2];holdout=group.iloc[1::2]
    offset=float(train.offset_frames.median())
    residual=(holdout.offset_frames-offset).abs()
    models.append(dict(half=half,offset_frames=offset,fit_anchors=len(train),check_anchors=len(holdout),
        holdout_median_error_seconds=float(residual.median()/FPS),
        holdout_p90_error_seconds=float(residual.quantile(.9)/FPS),
        holdout_within_040s_fraction=float((residual<=10).mean()),
        median_spatial_error_m=float(group.spatial_m.median())))
(OUT/'alignment_models.json').write_text(json.dumps(models,indent=2))
print(json.dumps(models,indent=2))


In [ ]:
# Map only time spans bracketed by consistent live-view anchors; no extrapolation.
# Переносим только участки между согласованными якорями прямого показа; без экстраполяции.
B=len(b['homography_valid']);T=len(t['homography_valid'])
mapped=np.full(B,-1,dtype=np.int32)
half_of_b=np.zeros(B,dtype=np.int8)
accepted_anchors=[]
for fitted in models:
    half=fitted['half'];offset=int(round(fitted['offset_frames']))
    group=anchors[(anchors.half==half)&(abs(anchors.offset_frames-offset)<=5)&(anchors.support>=.6)&(anchors.spatial_m<=2.5)].sort_values('broadcast_frame')
    accepted_anchors.extend(group.to_dict('records'))
    rr=group.broadcast_frame.to_numpy(dtype=int);dd=group.offset_frames.to_numpy(dtype=int)
    first,last=next((f,l) for h,f,l in half_video_blocks(GAME) if h==half)
    for j in range(len(rr)-1):
        if rr[j+1]-rr[j]>180*FPS or abs(dd[j+1]-dd[j])>10:continue
        frames=np.arange(rr[j],rr[j+1]+1)
        tactical=frames+offset
        ok=(tactical>=first)&(tactical<=last)&(frames>=0)&(frames<B)
        mapped[frames[ok]]=tactical[ok];half_of_b[frames[ok]]=half
pd.DataFrame({'broadcast_frame':np.arange(B),'tactical_frame':mapped,'half':half_of_b,
              'mapped':mapped>=0}).to_csv(OUT/'broadcast_to_tactical.csv',index=False)
# Missing homography is a scene-change proxy, NOT a true D label.
# Отсутствие гомографии — признак необычного плана, НЕ истинная метка D.
bad=~b['homography_valid']
starts=np.flatnonzero(bad&np.r_[True,~bad[:-1]])
stops=np.flatnonzero(bad&np.r_[~bad[1:],True])+1
accepted=pd.DataFrame(accepted_anchors)
intervals=[];weak_D=np.zeros(T,bool);mapped_nonfield=np.zeros(T,bool)
for a,z in zip(starts,stops):
    if z-a<FPS or np.any(mapped[a:z]<0):continue
    half=int(half_of_b[a])
    candidates=accepted[accepted.half==half].broadcast_frame.to_numpy()
    prev=candidates[candidates<a];following=candidates[candidates>=z]
    if not len(prev) or not len(following):continue
    # Nearby anchors on BOTH sides protect against edited-out time or replay alignment.
    # Близкие якоря с ОБЕИХ сторон защищают от вырезанного времени и привязки к повтору.
    if a-prev[-1]>30*FPS or following[0]-z>30*FPS:continue
    mapped_nonfield[mapped[a:z]]=True
    # User's 75% hypothesis, with 1-second margins; never represented as ground truth.
    # Гипотеза пользователя о первых 75% с отступом 1 с; это не эталон.
    safe_a=a+FPS;safe_z=min(a+int(.75*(z-a)),z-FPS)
    if safe_z<=safe_a:continue
    weak_D[mapped[safe_a:safe_z]]=True
    intervals.append(dict(broadcast_start=int(a),broadcast_end_exclusive=int(z),
        tactical_start=int(mapped[a]),tactical_end_exclusive=int(mapped[z-1]+1),
        weak_D_start=int(mapped[safe_a]),weak_D_end_exclusive=int(mapped[safe_z-1]+1),
        half=half,duration_s=(z-a)/FPS,weak_D_duration_s=(safe_z-safe_a)/FPS,
        status='hypothesis_75pct_not_ground_truth'))
pd.DataFrame(intervals).to_csv(OUT/'mapped_nonfield_intervals.csv',index=False)
np.save(OUT/'weak_D_candidates.npy',weak_D)
np.save(OUT/'mapped_nonfield_mask.npy',mapped_nonfield)

# Additional LIVE candidates require contemporaneous geometry AND movement, not just grass.
# Для LIVE нужны совпадающая текущая геометрия и движение, а не просто видимое поле.
geometry_live=np.zeros(T,bool)
for row in accepted.itertuples():
    a=max(0,row.broadcast_frame-2*FPS);z=min(B,row.broadcast_frame+2*FPS+1)
    frames=np.arange(a,z);valid=(mapped[frames]>=0)&b['homography_valid'][frames]
    geometry_live[mapped[frames[valid]]]=True
motion_path=ROOT/'local_review/experiments/008_weak_dead_boosting'
feature_names=json.loads((motion_path/'feature_names.json').read_text())
features=np.load(motion_path/'features.npy',mmap_mode='r')
col=lambda name:features[:,feature_names.index(name)]
weak_LIVE=geometry_live&(col('ball_speed_25')>=2)&(col('ball_speed_25')<=35)&(col('median_player_speed')>=.8)&~weak_D
np.save(OUT/'weak_LIVE_candidates.npy',weak_LIVE)
# These are exclusions for a future training run; this notebook trains no classifier.
# Исключение для будущего обучения; здесь никакой классификатор не обучается.
public_start,public_stop=11185,18685
training_allowed=np.ones(T,bool);training_allowed[max(0,public_start-20*FPS):public_stop+20*FPS]=False

def count_runs(mask):return int(np.count_nonzero(mask&np.r_[True,~mask[:-1]]))
summary=dict(geometric_anchors=len(anchors),accepted_anchors=len(accepted),
    mapped_broadcast_frames=int((mapped>=0).sum()),mapped_broadcast_percent=float((mapped>=0).mean()*100),
    mapped_invalid_intervals=len(intervals),
    weak_D_frames=int(weak_D.sum()),weak_D_minutes=float(weak_D.sum()/FPS/60),
    weak_LIVE_frames=int(weak_LIVE.sum()),weak_LIVE_minutes=float(weak_LIVE.sum()/FPS/60),
    training_eligible_weak_D_frames=int((weak_D&training_allowed).sum()),
    training_eligible_weak_D_runs=count_runs(weak_D&training_allowed),
    training_eligible_weak_LIVE_frames=int((weak_LIVE&training_allowed).sum()),
    no_ground_truth_used_for_alignment=True,labels_are_hypotheses=True)
# Inspect public overlap only AFTER all alignment and weak-label rules are fixed.
# Публичный эталон читаем только ПОСЛЕ фиксации синхронизации и гипотез меток.
gt=pd.read_csv(ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv')
public=gt.label.isin(['H','A','D']).to_numpy();true_D=gt.label.eq('D').to_numpy()
summary['public_probe']=dict(D_candidate_frames=int((weak_D&public).sum()),
    D_candidate_true_D=int((weak_D&public&true_D).sum()),
    D_candidate_precision=float((weak_D&public&true_D).sum()/max((weak_D&public).sum(),1)),
    true_D_coverage=float((weak_D&public&true_D).sum()/max((public&true_D).sum(),1)),
    LIVE_candidate_frames=int((weak_LIVE&public).sum()),
    LIVE_candidate_true_LIVE=int((weak_LIVE&public&~true_D).sum()),
    LIVE_candidate_precision=float((weak_LIVE&public&~true_D).sum()/max((weak_LIVE&public).sum(),1)))
(OUT/'mapping_summary.json').write_text(json.dumps(summary,indent=2))
lines=['# Broadcast → tactical: автоматическая синхронизация','',
    'Сопоставление геометрии без видео, ручных якорей или эталонных меток. Сначала кандидаты по мячу, затем расстановки команд без идентичности слотов. Уточнение — по тактическим кадрам 25 FPS.',
    'Два смещения времени соответствуют таймам. Проверка на чередующихся геометрических якорях — внутренняя оценка согласованности, не гарантия покадровой точности. Использовать с временными отступами.',
    'Перенос разрешён только между согласованными якорями с интервалом до 180 секунд. Необычные планы дополнительно требуют якорей не дальше 30 секунд с обеих сторон.',
    '', '## Согласованность по отложенным геометрическим якорям', '',
    '| Тайм | Якорей проверки | Медиана расхождения, с | 90-й процентиль, с |','|---|---:|---:|---:|']
for m in models:lines.append(f"| {m['half']} | {m['check_anchors']} | {m['holdout_median_error_seconds']:.3f} | {m['holdout_p90_error_seconds']:.3f} |")
lines+=['','## Кандидаты для слабой разметки','',json.dumps(summary,indent=2,ensure_ascii=False),
    '', 'Нет гомографии ≠ гарантированная остановка. Есть гомография ≠ гарантированная живая игра. Правило первых 75% — проверяемая гипотеза; кадры одной серии сильно зависимы.',
    'LIVE-кандидаты требуют дополнительно движения мяча и игроков, но движение возможно и при D. Совпадение повторов с прошлым действием исключается, когда оно противоречит основному смещению текущего тайма.',
    'Публичная проверка неоднократно использовалась раньше. Не менять правила по её результатам и не называть её независимой оценкой.',
    'Таблица соответствия и интервалы остаются локальными. Модели D/H/A не обучались и не изменялись. Для повторного Run All нужны локальные признаки 008 (или предварительный запуск его подготовки признаков).']
(OUT/'REVIEW_RU.md').write_text('\n'.join(lines)+'\n')
print(json.dumps(summary,indent=2))


In [ ]:
# Validate saved mapping and masks without accessing reference labels.
# Проверяем сохранённые соответствия и маски без обращения к эталону.
saved_mapping = pd.read_csv(OUT / 'broadcast_to_tactical.csv')
saved_d = np.load(OUT / 'weak_D_candidates.npy')
saved_live = np.load(OUT / 'weak_LIVE_candidates.npy')
saved_nonfield = np.load(OUT / 'mapped_nonfield_mask.npy')
assert len(saved_mapping) == 164488
assert saved_d.shape == saved_live.shape == saved_nonfield.shape == (199319,)
assert not np.any(saved_d & saved_live)
assert np.all(~saved_d | saved_nonfield)
covered = np.zeros(199319, dtype=bool)
for half_id, offset, first, last in [(1, 4014, 11185, 90651), (2, 26920, 110808, 190197)]:
    subset = saved_mapping[(saved_mapping['half'] == half_id) & saved_mapping['mapped']]
    indices = subset.tactical_frame.to_numpy(dtype=int)
    assert np.all((indices >= first) & (indices <= last))
    assert np.all(np.diff(indices) > 0)
    assert np.all(indices - subset.broadcast_frame.to_numpy() == offset)
    covered[indices] = True
assert np.all(~(saved_d | saved_live | saved_nonfield) | covered)
# Explicit training masks exclude the public window plus 20 seconds on both sides.
# Явные маски обучения исключают публичное окно и по 20 секунд с каждой стороны.
allowed = np.ones(199319, dtype=bool)
allowed[11185 - 500:18685 + 500] = False
np.save(OUT / 'training_weak_D_mask.npy', saved_d & allowed)
np.save(OUT / 'training_weak_LIVE_mask.npy', saved_live & allowed)
validation = {'mapping_bounds': True, 'monotonic_within_half': True,
              'D_LIVE_disjoint': True, 'all_candidates_mapped': True,
              'public_window_excluded_from_training_masks': True}
(OUT / 'validation.json').write_text(json.dumps(validation, indent=2))
print(validation)
